In [1]:
#imports
import numpy as np
import scipy.io
from scipy import signal as sig
from scipy.interpolate import CubicSpline
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter1d
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
# load training ECoG + data glove and leaderboard ECoG for all 3 subjects
train_data = scipy.io.loadmat('raw_training_data.mat')
lead_data = scipy.io.loadmat('leaderboard_data.mat')

train_ecog = [train_data['train_ecog'][i, 0].astype(np.float32) for i in range(3)]
train_dg = [train_data['train_dg'][i, 0].astype(np.float32)   for i in range(3)]
lead_ecog = [lead_data['leaderboard_ecog'][i, 0].astype(np.float32) for i in range(3)]

for i in range(3):
    print(f'Subject {i+1}: ECoG {train_ecog[i].shape}, DG {train_dg[i].shape}')
print('Leaderboard:', [e.shape for e in lead_ecog])

Subject 1: ECoG (300000, 62), DG (300000, 5)
Subject 2: ECoG (300000, 48), DG (300000, 5)
Subject 3: ECoG (300000, 64), DG (300000, 5)
Leaderboard: [(147500, 62), (147500, 48), (147500, 64)]


In [3]:
# Preprocessing: bandpass 0.5-200 Hz, remove outliers with MAD, notch filter 60/120 Hz
#then extract log band power features in 50ms windows across 11 frequency bands
def _get_bandpass(f_low, f_high, sample_rate=1000, filt_order=4):
    return sig.butter(filt_order, [f_low, f_high], btype='bandpass', fs=sample_rate, output='sos')

def filter_data(data, fs=1000):
    data = data.astype(np.float32)
    data = data - data.mean(axis=1, keepdims=True)
    channel_median = np.median(data, axis=0, keepdims=True)
    median_abs_dev = np.median(np.abs(data - channel_median), axis=0, keepdims=True) + 1e-6
    threshold = 6 * 1.4826 * median_abs_dev
    data = np.clip(data, channel_median - threshold, channel_median + threshold)
    cleaned = sig.sosfiltfilt(_get_bandpass(0.5, 200, sample_rate=fs, filt_order=4), data, axis=0)

    for notch_freq in [60, 120]:
        b, a = sig.iirnotch(notch_freq, Q=30, fs=fs)
        cleaned = sig.filtfilt(b, a, cleaned, axis=0)
    return cleaned.astype(np.float32)

FREQ_BANDS = [
    (0.5, 4), (4, 8), (8, 13), (13, 30), (30, 50),
    (50, 70), (70, 90), (90, 110), (110, 130),
    (130, 150), (150, 170)
]

def compute_band_features(filtered, fs=1000, frame_ms=50):
    win_len = int(fs * frame_ms / 1000)
    n_samples, num_channels = filtered.shape
    num_frames = n_samples // win_len
    windowed = filtered[:num_frames * win_len].reshape(num_frames, win_len, num_channels)
    feature_list = [windowed.mean(axis=1)]

    for f_low, f_high in FREQ_BANDS:
        bp_sos = _get_bandpass(f_low, f_high, sample_rate=fs, filt_order=4)
        band_signal = sig.sosfiltfilt(bp_sos, filtered, axis=0)
        band_signal = band_signal[:num_frames * win_len].reshape(num_frames, win_len, num_channels)
        feature_list.append(np.log(np.mean(band_signal ** 2, axis=1) + 1e-4))
    all_features = np.concatenate(feature_list, axis=1).astype(np.float32)

    return np.nan_to_num(all_features, nan=0.0, posinf=0.0, neginf=0.0), win_len

In [4]:
# build sliding window sequences (30 frames of context) for the model input
# also downsamples the dg to match the feature frame rate during training
CONTEXT_FRAMES = 30
FRAME_SAMPLES = 50

def build_sequences(features, dg=None, context=CONTEXT_FRAMES):
    n_frames, F = features.shape
    pad = np.repeat(features[:1],context - 1, axis=0)
    padded = np.concatenate([pad, features], axis=0)
    X = np.lib.stride_tricks.sliding_window_view(
            padded, window_shape=context, axis=0
        ).transpose(0, 2, 1)[:n_frames]
    if dg is not None:
        Y = np.zeros((n_frames, 5), dtype=np.float32)
        for i in range(n_frames):
            s = i * FRAME_SAMPLES
            Y[i] = dg[s:s+FRAME_SAMPLES].mean(axis=0)
        return X.astype(np.float32), Y
    return X.astype(np.float32)

In [5]:
# two model architectures: ResNet-style CNN and CNN+BiGRU hybrid
# both output 5 values, one per finger, and use Huber loss
def residual_block(inp, n_filters, kernel=3, drop_rate=0.0):
    r = layers.Conv1D(n_filters, kernel, padding='same', use_bias=False)(inp)
    r = layers.BatchNormalization()(r)
    r = layers.ReLU()(r)

    if drop_rate > 0:
        r = layers.SpatialDropout1D(drop_rate)(r)
    r = layers.Conv1D(n_filters, kernel, padding='same', use_bias=False)(r)
    r = layers.BatchNormalization()(r)

    if inp.shape[-1] != n_filters:
        inp = layers.Conv1D(n_filters, 1, padding='same', use_bias=False)(inp)
    return layers.ReLU()(layers.Add()([inp, r]))

def build_cnn(n_features):
    input_layer = keras.Input(shape=(CONTEXT_FRAMES, n_features))
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(input_layer)
    x = layers.BatchNormalization()(x)
    x = residual_block(x, 128)
    x = residual_block(x, 192)
    x = layers.Dropout(0.2)(x)
    x = residual_block(x, 256)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(5)(x)
    model = keras.Model(input_layer, output)
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=8e-4, weight_decay=1e-4, clipnorm=1.0),
        loss='huber'
    )
    return model

def build_cnn_gru(n_features):
    input_layer = keras.Input(shape=(CONTEXT_FRAMES, n_features))
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(input_layer)
    x = layers.BatchNormalization()(x)
    x = residual_block(x, 128, drop_rate=0.15)
    x = residual_block(x, 192, drop_rate=0.15)
    x = layers.Dropout(0.35)(x)
    x = residual_block(x, 256, drop_rate=0.15)
    x = layers.Bidirectional(layers.GRU(96, return_sequences=False, dropout=0.2))(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.45)(x)
    output = layers.Dense(5)(x)
    model = keras.Model(input_layer, output)
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=8e-4, weight_decay=1e-4, clipnorm=1.0),
        loss='huber'
    )
    return model


In [6]:
# train 10 models per subject (5 seeds x 2 architectures), keep the best 3
# evaluate using pearson correlation on fingers 1,2,3,5 skip 4
SEEDS_PER_ARCH = 5
KEEP_TOP = 3
EPOCHS = 70
BATCH = 256
VAL_FRAC = 0.05

ensembles = []

for subj in range(3):
    print(f'\n--- Subject {subj+1} ---')
    filtered = filter_data(train_ecog[subj])
    F, _ = compute_band_features(filtered)
    print(f'  Features shape: {F.shape}')

    T_frames = F.shape[0]
    split = int((1 - VAL_FRAC) * T_frames)
    f_mean = F.mean(0, keepdims=True)
    f_std = F.std(0,  keepdims=True) + 1e-6
    F_norm = (F - f_mean) / f_std

    X, Y = build_sequences(F_norm, train_dg[subj])
    X_tr,Y_tr = X[:split], Y[:split]
    X_va, Y_va = X[split:], Y[split:]

    y_mean = Y_tr.mean(0, keepdims=True)
    y_std = Y_tr.std(0,  keepdims=True) + 1e-6
    Y_tr_n = (Y_tr - y_mean) / y_std
    Y_va_n = (Y_va - y_mean) / y_std

    all_models, all_scores, all_names = [], [], []

    for arch_name, builder in [('cnn', build_cnn), ('cnn_gru', build_cnn_gru)]:
        for seed in range(SEEDS_PER_ARCH):
            tag = f'{arch_name}-s{seed}'
            print(f'  -- training {tag}')
            keras.utils.set_random_seed(seed + (0 if arch_name == 'cnn' else 100))
            m = builder(F.shape[1])
            cbs = [
                keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss'),
                keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=6,min_lr=1e-5),
            ]
            m.fit(X_tr, Y_tr_n, validation_data=(X_va, Y_va_n),
                  epochs=EPOCHS, batch_size=BATCH, callbacks=cbs, verbose=0)
            p = m.predict(X_va, verbose=0) * y_std + y_mean
            corrs = [pearsonr(Y_va[:, f], p[:, f])[0] for f in range(5)]
            score = np.mean([corrs[0], corrs[1], corrs[2], corrs[4]])
            print(f'    {tag}  corrs={corrs}  avg={score:.3f}')
            all_models.append(m); all_scores.append(score); all_names.append(tag)

    order = np.argsort(all_scores)[:: -1][:KEEP_TOP]
    kept_models = [all_models[i] for i in order]
    kept_names = [all_names[i]  for i in order]
    print(f'\n  Kept top-{KEEP_TOP}: {kept_names}')

    preds = np.mean([m.predict(X_va, verbose=0) for m in kept_models], axis=0) * y_std + y_mean
    corrs = [pearsonr(Y_va[:, f], preds[:, f])[0] for f in range(5)]
    score = np.mean([corrs[0], corrs[1], corrs[2], corrs[4]])
    print(f'  Ensemble corrs: {corrs}  avg={score:.3f}')

    ensembles.append({'models': kept_models, 'f_mean': f_mean, 'f_std': f_std, 'y_mean': y_mean, 'y_std': y_std})


--- Subject 1 ---
  Features shape: (6000, 744)
  -- training cnn-s0
    cnn-s0  corrs=[np.float32(0.59451765), np.float32(0.7186407), np.float32(0.15203725), np.float32(0.57254153), np.float32(0.6534912)]  avg=0.530
  -- training cnn-s1
    cnn-s1  corrs=[np.float32(0.31120557), np.float32(0.68936306), np.float32(0.41651642), np.float32(0.56329864), np.float32(0.26539028)]  avg=0.421
  -- training cnn-s2
    cnn-s2  corrs=[np.float32(0.51387525), np.float32(0.74762714), np.float32(0.43806896), np.float32(0.6046876), np.float32(0.11575222)]  avg=0.454
  -- training cnn-s3
    cnn-s3  corrs=[np.float32(0.43856168), np.float32(0.775928), np.float32(0.41905236), np.float32(0.5372295), np.float32(0.208156)]  avg=0.460
  -- training cnn-s4
    cnn-s4  corrs=[np.float32(0.44137484), np.float32(0.7526249), np.float32(0.38014582), np.float32(0.5360056), np.float32(0.28345317)]  avg=0.464
  -- training cnn_gru-s0
    cnn_gru-s0  corrs=[np.float32(0.5357926), np.float32(0.65207577), np.float32(

In [7]:
# save trained models and normalization params
import os, zipfile, pickle

os.makedirs('saved_models', exist_ok=True)
save_data = []

for subj, ens in enumerate(ensembles):
    subj_dir = f'saved_models/subject{subj+1}'
    os.makedirs(subj_dir, exist_ok=True)
    model_paths = []
    for mi, m in enumerate(ens['models']):
        path = f'{subj_dir}/model_{mi}.keras'
        m.save(path)
        model_paths.append(path)
        print(f'  Saved subject {subj+1} model {mi}: {path}')
    save_data.append({
        'model_paths': model_paths,
        'f_mean': ens['f_mean'], 'f_std': ens['f_std'],
        'y_mean': ens['y_mean'], 'y_std': ens['y_std'],
    })

with open('ensemble_params.pkl', 'wb') as f:
    pickle.dump(save_data, f)
print('Saved ensemble_params.pkl')
with zipfile.ZipFile('submission_models.zip','w', zipfile.ZIP_DEFLATED) as zf:
    for subj in range(3):
        for mi in range(len(ensembles[subj]['models'])):
            path = f'saved_models/subject{subj+1}/model_{mi}.keras'
            zf.write(path)
print('Created submission_models.zip')

  Saved subject 1 model 0: saved_models/subject1/model_0.keras
  Saved subject 1 model 1: saved_models/subject1/model_1.keras
  Saved subject 1 model 2: saved_models/subject1/model_2.keras
  Saved subject 2 model 0: saved_models/subject2/model_0.keras
  Saved subject 2 model 1: saved_models/subject2/model_1.keras
  Saved subject 2 model 2: saved_models/subject2/model_2.keras
  Saved subject 3 model 0: saved_models/subject3/model_0.keras
  Saved subject 3 model 1: saved_models/subject3/model_1.keras
  Saved subject 3 model 2: saved_models/subject3/model_2.keras
Saved ensemble_params.pkl
Created submission_models.zip


In [8]:
# download locally
from google.colab import files
files.download('submission_models.zip')
files.download('ensemble_params.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
#run ensemble on leaderboard data, smooth predictions, and interpolate back to 1000 Hz using cubic splines
SMOOTH_SIGMAS = [4.0, 3.0, 2.5, 3.0, 4.0]
predicted_dg = []

for subj in range(3):
    ens = ensembles[subj]
    filt = filter_data(lead_ecog[subj])
    feat, frame_len = compute_band_features(filt)

    feat_norm = np.nan_to_num((feat - ens['f_mean']) / ens['f_std'], nan=0.0, posinf=0.0, neginf=0.0)
    X = build_sequences(feat_norm)

    raw_preds = np.mean([m.predict(X, batch_size=512, verbose=0) for m in ens['models']], axis=0)
    raw_preds = raw_preds * ens['y_std'] + ens['y_mean']
    raw_preds = np.nan_to_num(raw_preds, nan=0.0, posinf=0.0, neginf=0.0)

    # smooth each finger's prediction
    smoothed = np.stack([gaussian_filter1d(raw_preds[:, f], sigma=SMOOTH_SIGMAS[f]) for f in range(5)], axis=1)

    n_lead_samples = filt.shape[0]
    frame_centers = np.arange(smoothed.shape[0]) * frame_len + frame_len / 2
    t_out = np.arange(n_lead_samples)
    interp_dg = np.zeros((n_lead_samples, 5), dtype=np.float32)

    for i in range(5):
        t_nodes = np.concatenate([[0], frame_centers, [n_lead_samples - 1]])
        y_nodes = np.concatenate([[smoothed[0, i]], smoothed[:,i], [smoothed[-1, i]]])
        cs = CubicSpline(np.nan_to_num(t_nodes), np.nan_to_num(y_nodes), bc_type='not-a-knot')
        interp_dg[:, i] = cs(t_out).astype(np.float32)

    predicted_dg.append(interp_dg)
    print(f'Subject {subj+1}: {interp_dg.shape}')


Subject 1: (147500, 5)
Subject 2: (147500, 5)
Subject 3: (147500, 5)


In [10]:
# save predictions  as a 3x1 cell array .mat file for leaderboard submission
out = np.empty((3, 1), dtype=object)
for i in range(3):
    out[i, 0] = predicted_dg[i]

scipy.io.savemat('predicted_dg.mat', {'predicted_dg': out})
print('Saved predicted_dg.mat')

Saved predicted_dg.mat
